# Traffic Sign Detection v3
One semantic model; global + sliced inference; geometry-first fusion; crop re-check; HSV/template validation; class-agnostic temporal voting. Uncertain distant detections are shown as `Sign` instead of forcing a wrong class. Select **T4 GPU** and **Run all**.

In [ ]:
!pip install -q "ultralytics>=8.3,<9" "huggingface_hub>=0.25" "opencv-python-headless>=4.9" "requests>=2.31" "tqdm>=4.66"
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import defaultdict,deque
from dataclasses import dataclass,field
import cv2,numpy as np,torch,requests,subprocess,shutil,math,re,csv
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

R=Path('/content/drive/MyDrive/DIP'); V=R/'video1.mp4'; M=R/'models'; O=R/'outputs'
MD=M/'traffic_sign_yolo11s'; T=M/'sign_templates_v3'
for p in (M,O,MD,T):p.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(),'Enable T4 GPU first'; assert V.exists(),f'Missing {V}'
for p in [M/x for x in ['traffic_sign_primary','traffic_sign_secondary','traffic_sign','sign_templates','helmet','license_plate','scene','easyocr']]+[O/'video1_plates',O/'video1_violations']:
    if p.exists():shutil.rmtree(p,ignore_errors=True)
for p in [M/'helmet_best.pt',M/'plate_best.pt',O/'video1_result.mp4',O/'video1_result.csv',O/'video1_result_temp.mp4',O/'video1_audit.jpg']:
    try:p.unlink()
    except FileNotFoundError:pass
mp=hf_hub_download('star092304/traffic-sign-detection-vietnam-yolo','best.pt',local_dir=str(MD))
model=YOLO(mp); DEV=0
print('GPU:',torch.cuda.get_device_name(0),'\nModel:',mp)

OFF={'No Entry':'P102','No Stopping & No Parking':'P130','No Parking':'P131a','No Left Turn':'P123a','No Right Turn':'P123b','No U-Turn':'P124a1','Turn Right Only':'R302a','Keep left':'R302b','Roundabout':'R303','Children Crossing':'W225','Road Work Ahead':'W227','No Overtaking':'P125'}
def sn(s):return re.sub(r'[^A-Za-z0-9_-]+','_',s).strip('_')
for lab,c in OFF.items():
    out=T/f'{sn(lab)}.png'
    if out.exists() and out.stat().st_size>500:continue
    try:
        q={'action':'query','format':'json','prop':'imageinfo','iiprop':'url','iiurlwidth':320,'titles':f'File:Vietnam road sign {c}.svg'}
        d=requests.get('https://commons.wikimedia.org/w/api.php',params=q,timeout=20,headers={'User-Agent':'TDTU-DIP/3'}).json()
        info=next(iter(d['query']['pages'].values()))['imageinfo'][0];u=info.get('thumburl') or info['url']
        z=requests.get(u,timeout=20,headers={'User-Agent':'TDTU-DIP/3'});z.raise_for_status();out.write_bytes(z.content)
    except Exception as e:print('[template optional]',lab,e)
print('Templates:',len(list(T.glob('*.png'))))


In [ ]:
D={'Turn Right Only':'Keep Right','No Stopping & No Parking':'No Stop/Parking','Children Crossing':'Children','Road Work Ahead':'Road Works','One way street':'One Way','Road with Surveillance Camera':'Camera Ahead','No U-Turn and No Left Turn':'No U-Turn/Left','No U-Turn and No Right Turn':'No U-Turn/Right','No U-Turn and Left Turn for Cars':'No U-Turn/Left (Cars)','Double curve first to right':'Double Curve Right','sparsely populated area':'Sparse Area'}
X={'Green Light','Red Light'}
BM={'Turn Right Only','Keep left','Turn Left','Turn Right','Roundabout','One way street','Parking','Bus Stop','Hospital','U-Turn Area'}
BR={'No Stopping & No Parking','No Parking','No Parking on Odd Days','No Parking on Even Days'}
WA={'Danger','Slow Down','Double curve first to right','Obstacle on the Road','Traffic light ahead','Sharp Left Turn','Sharp Right Turn','Children Crossing','Pedestrian Crossing','Road Work Ahead','Accident area','Steep ascent','Narrow bridge','Uneven road','Slippery Road'}
@dataclass
class RD:b:tuple;r:str;c:float;s:str
@dataclass
class OD:b:tuple;r:str;l:str;c:float;q:float;p:int;a:float;t:float;px:float

def dl(r):return D.get(r,r[:28])
def io(a,b):
 x=max(a[0],b[0]);y=max(a[1],b[1]);X=min(a[2],b[2]);Y=min(a[3],b[3]);i=max(0,X-x)*max(0,Y-y)
 if i<=0:return 0.
 A=max(1,a[2]-a[0])*max(1,a[3]-a[1]);B=max(1,b[2]-b[0])*max(1,b[3]-b[1]);return i/(A+B-i)
def cm(a,b):
 ax=(a[0]+a[2])/2;ay=(a[1]+a[3])/2;bx=(b[0]+b[2])/2;by=(b[1]+b[3])/2
 sa=math.sqrt(max(1,(a[2]-a[0])*(a[3]-a[1])));sb=math.sqrt(max(1,(b[2]-b[0])*(b[3]-b[1])))
 return math.hypot(ax-bx,ay-by)/max(8,(sa+sb)/2),max(sa,sb)/max(1,min(sa,sb))
def same(a,b):
 if io(a,b)>=.28:return True
 d,s=cm(a,b);tiny=max(a[2]-a[0],a[3]-a[1],b[2]-b[0],b[3]-b[1])<=42
 return tiny and d<=.46 and s<=2.1
def okbox(b,W,H):
 w=max(1,b[2]-b[0]);h=max(1,b[3]-b[1]);return min(w,h)>=5 and .18<=w/h<=5.5 and w*h/(W*H)<.10 and (b[1]+b[3])/(2*H)<.94
def hs(c):
 if c.size==0:return (0,0,0,0,0,0)
 h=cv2.cvtColor(c,cv2.COLOR_BGR2HSV);H,S,V=cv2.split(h)
 return (((((H<=12)|(H>=165))&(S>=55)&(V>=40)).mean()),(((H>=88)&(H<=138)&(S>=45)&(V>=35)).mean()),(((H>=14)&(H<=42)&(S>=45)&(V>=45)).mean()),(((S<=55)&(V>=120)).mean()),V.mean(),V.std())
def ap(r,s):
 R,B,Y,W,_,_=s
 if r=='No Entry':return -.85 if B>.18 and B>R*1.45 else float(np.clip(2.8*R+W-1.5*B-.25,-1,1))
 if r in BM:return -.65 if R>.24 and R>B*1.8 else float(np.clip(3*B-R-.15,-1,1))
 if r in BR:return float(np.clip(1.8*min(.35,R)+1.8*min(.35,B)-.2,-1,1))
 if r.startswith('Speed limit') or r.startswith('No ') or r in {'Low Clearance','Height Limit','No Overtaking'}:
  return -.5 if B>.28 and R<.05 else float(np.clip(2.4*R-.1,-1,1))
 if r in WA:return float(np.clip(1.8*R+1.2*Y-.12,-1,1))
 return 0.
def enh(x):
 L,a,b=cv2.split(cv2.cvtColor(x,cv2.COLOR_BGR2LAB))
 if L.mean()>=75 and L.std()>=28:return x
 L=cv2.createCLAHE(1.5 if L.mean()>=55 else 2,(8,8)).apply(L);return cv2.cvtColor(cv2.merge([L,a,b]),cv2.COLOR_LAB2BGR)
def tiles(f):
 H,W=f.shape[:2];y=int(H*.79);w=int(W*.5);return [(f[:y,x:x+w],(x,0),f's{i}') for i,x in enumerate([0,int(W*.25),W-w])]
def inf(im,off=(0,0),src='g',cf=.18,sz=704):
 ox,oy=off;H,W=SH
 z=model.predict(im,conf=cf,iou=.55,imgsz=sz,device=DEV,verbose=False,max_det=60,agnostic_nms=True)[0];out=[]
 if z.boxes is None:return out
 for b in z.boxes:
  k=int(b.cls[0]);r=str(z.names.get(k,k) if isinstance(z.names,dict) else z.names[k])
  if r in X:continue
  x1,y1,x2,y2=map(int,b.xyxy[0].tolist());bb=(max(0,x1+ox),max(0,y1+oy),min(W-1,x2+ox),min(H-1,y2+oy))
  if okbox(bb,W,H):out.append(RD(bb,r,float(b.conf[0]),src))
 return out

TP=defaultdict(list)
for p in T.glob('*.png'):
 im=cv2.imread(str(p))
 if im is None:continue
 k=next((r for r in OFF if p.stem==sn(r)),None)
 if k:
  e=cv2.Canny(cv2.cvtColor(cv2.resize(im,(72,72)),cv2.COLOR_BGR2GRAY),55,150).astype('float32');TP[k].append((e-e.mean())/(e.std()+1e-6))
def ts(c,r):
 if r not in TP or c.size==0:return 0.
 e=cv2.Canny(cv2.cvtColor(cv2.resize(c,(72,72)),cv2.COLOR_BGR2GRAY),55,150).astype('float32');e=(e-e.mean())/(e.std()+1e-6)
 return max(float((e*t).mean()) for t in TP[r])
def crop(f,b):
 H,W=f.shape[:2];x1,y1,x2,y2=b;cx=(x1+x2)/2;cy=(y1+y2)/2;s=min(int(min(H,W)*.55),max(96,int(max(x2-x1,y2-y1)*4.2)))
 a=max(0,int(cx-s/2));q=max(0,int(cy-s/2));A=min(W,int(cx+s/2));Q=min(H,int(cy+s/2));return f[q:Q,a:A]
def refine(f,b):
 c=enh(crop(f,b))
 if c.size==0:return None
 z=model.predict(c,conf=.12,iou=.55,imgsz=512,device=DEV,verbose=False,max_det=12,agnostic_nms=True)[0]
 if z.boxes is None:return None
 H,W=c.shape[:2];best=None;bs=-1
 for b in z.boxes:
  k=int(b.cls[0]);r=str(z.names.get(k,k) if isinstance(z.names,dict) else z.names[k])
  if r in X:continue
  x1,y1,x2,y2=map(float,b.xyxy[0]);d=math.hypot(((x1+x2)/2-W/2)/W,((y1+y2)/2-H/2)/H)
  sc=float(b.conf[0])*(1-d)
  if d<.34 and sc>bs:best=(r,float(b.conf[0]));bs=sc
 return best
def fuse(f,ds):
 C=[]
 for d in sorted(ds,key=lambda x:x.c,reverse=True):
  j=next((i for i,c in enumerate(C) if same(d.b,c[0].b)),None)
  C.append([d]) if j is None else C[j].append(d)
 rec=[]
 for c in C:
  w=np.array([x.c for x in c]);B=np.array([x.b for x in c]);bb=tuple(map(int,np.round((B*w[:,None]).sum(0)/w.sum())))
  v=defaultdict(float);ss=set()
  for x in c:v[x.r]+=x.c*(1 if x.s=='g' else .96);ss.add(x.s)
  top=max(v,key=v.get);rat=v[top]/(sum(v.values())+1e-9);co=min(.99,max(x.c for x in c)+.045*(len(ss)-1));x1,y1,x2,y2=bb
  st=hs(f[y1:y2,x1:x2]);rec.append([bb,v,len(ss),co,min(x2-x1,y2-y1),rat,ap(top,st)])
 cand=sorted([r for r in rec if (r[5]<.72 or r[6]<-.22) and (r[3]>=.22 or r[2]>=2)],key=lambda r:(r[2],r[3]),reverse=True)[:2]
 ids={id(x) for x in cand};out=[]
 for r in rec:
  bb,v,P,co,px,_,_=r
  if id(r) in ids:
   q=refine(f,bb)
   if q:v[q[0]]+=q[1]*1.15
  x1,y1,x2,y2=bb;c=f[y1:y2,x1:x2];st=hs(c);adj={};tt={}
  for lab,val in v.items():
   a=ap(lab,st);fac=.55 if a<=-.7 else (.75 if a<=-.45 else 1+.08*max(0,a));t=ts(c,lab) if len(v)>1 else 0;adj[lab]=val*fac*(1+.035*max(0,t));tt[lab]=t
  raw=max(adj,key=adj.get);q=adj[raw]/(sum(adj.values())+1e-9)
  out.append(OD(bb,raw,dl(raw),co,q,P,ap(raw,st),tt.get(raw,0),px))
 return out


In [ ]:
@dataclass
class TR:
 i:int;b:tuple;last:int;h:int=0;v:deque=field(default_factory=lambda:deque(maxlen=12));lock:str|None=None;ch:str|None=None;n:int=0
class Tracker:
 def __init__(s):s.t={};s.n=1
 def upd(s,ds,k):
  used=set();out=[]
  for d in sorted(ds,key=lambda x:x.c,reverse=True):
   z=None;bq=-1
   for i,t in s.t.items():
    if i in used or k-t.last>5 or not same(d.b,t.b):continue
    q=io(d.b,t.b)+(.2 if cm(d.b,t.b)[0]<.35 else 0)
    if q>bq:z=i;bq=q
   if z is None:z=s.n;s.n+=1;s.t[z]=TR(z,d.b,k)
   t=s.t[z];used.add(z);bb=tuple(int(.72*a+.28*b) for a,b in zip(d.b,t.b)) if t.h and io(d.b,t.b)>=.12 else d.b
   t.b=bb;t.last=k;t.h+=1;t.v.append((d.r,d.c*max(.25,d.q)*(1 if d.a>=-.45 else .35)))
   V=defaultdict(float)
   for a,b in t.v:V[a]+=b
   win=max(V,key=V.get);rr=V[win]/(sum(V.values())+1e-9)
   if t.lock is None:
    mn=2 if d.c>=.55 and d.q>=.72 else 3
    if t.h>=mn and rr>=.68 and d.a>=-.55:t.lock=win
   elif win!=t.lock:
    if t.ch==win:t.n+=1
    else:t.ch=win;t.n=1
    lw=V.get(t.lock,0)
    if t.n>=3 and V[win]>=max(lw*1.22,lw+.35):t.lock=win;t.ch=None;t.n=0
   else:t.ch=None;t.n=0
   spec=t.lock is not None and rr>=.66 and d.q>=.50 and d.c>=.23 and (d.p>=2 or d.c>=.46) and d.a>=-.55
   gen=not spec and t.h>=2 and d.px>=7 and ((d.p>=2 and d.c>=.26) or d.c>=.42)
   if spec:out.append((z,bb,dl(t.lock),'specific',d,rr))
   elif gen:out.append((z,bb,'Sign','generic',d,rr))
  for i in list(s.t):
   if k-s.t[i].last>20:del s.t[i]
  return out
def draw(f,b,txt,g=False):
 x1,y1,x2,y2=map(int,b);H,W=f.shape[:2];col=(145,145,145) if g else (0,155,255);cv2.rectangle(f,(x1,y1),(x2,y2),col,2,cv2.LINE_AA)
 sc=.43 if W<=1280 else .52;(tw,th),ba=cv2.getTextSize(txt,cv2.FONT_HERSHEY_SIMPLEX,sc,1)
 if x2-x1<45 and x2+tw+10<W:tx=x2+5;ty=max(th+6,min(H-4,y1+th))
 else:tx=max(2,min(W-tw-8,x1));ty=max(th+7,y1-5)
 ov=f.copy();cv2.rectangle(ov,(tx-3,ty-th-5),(tx+tw+4,ty+ba+3),(20,20,20),-1);cv2.addWeighted(ov,.72,f,.28,0,f)
 cv2.putText(f,txt,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,sc,(220,220,220) if g else (255,205,70),1,cv2.LINE_AA)

cap=cv2.VideoCapture(str(V));FPS=cap.get(cv2.CAP_PROP_FPS) or 30;N=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));W=int(cap.get(3));H=int(cap.get(4));SH=(H,W)
ok,test=cap.read();assert ok
_=inf(test);[_ for tile,off,name in tiles(test) for _ in [inf(enh(tile),off,name,.12,768)]]
print('Smoke test OK:',W,'x',H,'frames',N);cap.set(cv2.CAP_PROP_POS_FRAMES,0)
OUT=O/'video1_result.mp4';TMP=O/'video1_result_temp.mp4';CSV=O/'video1_result.csv';AUD=O/'video1_audit.jpg'
wr=cv2.VideoWriter(str(TMP),cv2.VideoWriter_fourcc(*'mp4v'),FPS,(W,H));assert wr.isOpened()
F=['frame','time_sec','track_id','label','raw_class','mode','confidence','class_ratio','temporal_ratio','passes','appearance','template','x1','y1','x2','y2']
ai={min(N-1,int(t*FPS)):t for t in [5,15,25,35,42,50,60,68,75,85,95,101]};A=[];tr=Tracker()
with open(CSV,'w',newline='',encoding='utf-8') as fh:
 lg=csv.DictWriter(fh,fieldnames=F);lg.writeheader();bar=tqdm(total=N,desc='v3: slices → fusion → semantic vote',unit='frame');k=0
 while True:
  ok,f=cap.read()
  if not ok:break
  org=f.copy();raw=inf(f)
  for tile,off,name in tiles(f):raw+=inf(enh(tile),off,name,.12,768)
  for tid,b,l,mo,d,rr in tr.upd(fuse(org,raw),k):
   draw(f,b,l,mo=='generic');x1,y1,x2,y2=b;lg.writerow(dict(frame=k,time_sec=round(k/FPS,3),track_id=tid,label=l,raw_class=d.r,mode=mo,confidence=round(d.c,4),class_ratio=round(d.q,4),temporal_ratio=round(rr,4),passes=d.p,appearance=round(d.a,4),template=round(d.t,4),x1=x1,y1=y1,x2=x2,y2=y2))
  if k in ai:
   a=cv2.resize(org,(640,360));b=cv2.resize(f,(640,360));cv2.putText(a,f'ORIGINAL {ai[k]}s',(10,28),0,.65,(255,255,255),2);cv2.putText(b,f'RESULT {ai[k]}s',(10,28),0,.65,(255,255,255),2);A.append(np.hstack([a,b]))
  wr.write(f);k+=1;bar.update(1)
 bar.close()
cap.release();wr.release()
base=['-c:v','libx264','-preset','veryfast','-crf','22','-pix_fmt','yuv420p','-tag:v','avc1','-movflags','+faststart']
try:subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(TMP),'-i',str(V),'-map','0:v:0','-map','1:a?',*base,'-c:a','aac','-shortest',str(OUT)],check=True)
except:subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(TMP),*base,'-an',str(OUT)],check=True)
try:TMP.unlink()
except:pass
if A:cv2.imwrite(str(AUD),np.vstack(A))
print('Saved:',OUT,'\nCSV:',CSV,'\nAudit:',AUD)
P=Path('/content/video1_result_preview.mp4');subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(OUT),'-vf','scale=960:-2','-c:v','libx264','-crf','29','-pix_fmt','yuv420p','-an',str(P)],check=True)
from IPython.display import Video,display
display(Video(str(P),embed=True,width=960,html_attributes='controls'))
